In [1]:
import sys
print(sys.executable)


d:\ML\venv310\Scripts\python.exe


In [2]:
!pip install sentence-transformers



[notice] A new release of pip is available: 23.0.1 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
!pip install "transformers>=4.41.0,<6.0.0"


[notice] A new release of pip is available: 23.0.1 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import numpy as np
import pickle
import json
import os

from sentence_transformers import SentenceTransformer


OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "d:\ML\venv310\lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [5]:
# Confidence thresholds (frozen defaults)
RC_THRESHOLD = 0.80
SUBCAT_THRESHOLD = 0.85
FAULT_THRESHOLD = 0.90


In [6]:
PHASE1_DIR = r"D:\ML\SM Modeling\Artifacts\Phase1"
PHASE2_DIR = r"D:\ML\SM Modeling\Artifacts\Phase2"
PHASE3_DIR = r"D:\ML\SM Modeling\Artifacts\Phase3"


Load All Artifacts

In [7]:
# Phase 1
rescode_clf = pickle.load(open(
    os.path.join(PHASE1_DIR, "rescode_embedding_clf.pkl"), "rb"
))
rescode_le = pickle.load(open(
    os.path.join(PHASE1_DIR, "rescode_label_encoder.pkl"), "rb"
))

embedding_model_name = open(
    os.path.join(PHASE1_DIR, "embedding_model_name.txt")
).read().strip()

# Phase 2
subcat_clf = pickle.load(open(
    os.path.join(PHASE2_DIR, "subcat_embedding_clf.pkl"), "rb"
))
subcat_le = pickle.load(open(
    os.path.join(PHASE2_DIR, "subcat_label_encoder.pkl"), "rb"
))

rescode_to_subcat = json.load(open(
    os.path.join(PHASE2_DIR, "rescode_to_subcat_map.json")
))

# Phase 3
fault_clf = pickle.load(open(
    os.path.join(PHASE3_DIR, "fault_embedding_clf.pkl"), "rb"
))
fault_le = pickle.load(open(
    os.path.join(PHASE3_DIR, "fault_label_encoder.pkl"), "rb"
))

rc_subcat_to_fault = json.load(open(
    os.path.join(PHASE3_DIR, "rc_subcat_to_fault_map.json")
))

embedder = SentenceTransformer(embedding_model_name)


NameError: name 'SentenceTransformer' is not defined

Load Input Data (Excel or CSV)

In [ ]:
df = pd.read_csv(
    r"D:\ML\SM Modeling\Data\Raw\incident_enriched.csv"
)

df = df[[
    "incident_number",
    "short_description",
    "task_text",
    "close_notes"
]]

df.to_excel(
    r"D:\ML\SM Modeling\Data\Input\incidents_input.xlsx",
    index=False
)

print("Test input file created")


Test input file created


In [ ]:
INPUT_PATH = r"D:\ML\SM Modeling\Data\Input\incidents_input.xlsx"

df = pd.read_excel(INPUT_PATH)
df.head()


,incident_number,short_description,task_text,close_notes
0,NaN,Proactive - S1206097  TLOS 4th Utility ISP...,NaN,Our systems have seen the affected service(s) ...
1,NaN,Proactive - S1276749  TLOS  No One Interne...,NaN,Our systems have seen the affected service(s) ...
2,NaN,Proactive - S1283402 TLOS  Sky UK Limited ...,NaN,No response from isp
3,NaN,Proactive - S1291431  TLOS  Sky UK Limited...,NaN,No availability provided
4,NaN,Proactive - S1341308  TLOS  4th Utility IS...,NaN,Resolving pending response\n\n


Clean Text & Build Semantic Text

In [ ]:
text_cols = ["short_description", "task_text", "close_notes"]

for col in text_cols:
    df[col] = df[col].fillna("").astype(str)

df["semantic_text"] = (
    df["short_description"].str.strip() + " " +
    df["task_text"].str.strip() + " " +
    df["close_notes"].str.strip()
)


Generate Embeddings (ONCE)

In [ ]:
X_emb = embedder.encode(
    df["semantic_text"].values,
    show_progress_bar=True,
    convert_to_numpy=True
)


NameError: name 'embedder' is not defined

Phase 1 Inference (Resolution Code)

In [ ]:
rc_probs = rescode_clf.predict_proba(X_emb)
df["rc_confidence"] = np.max(rc_probs, axis=1)

rc_idx = np.argmax(rc_probs, axis=1)
df["predicted_resolution_code"] = rescode_le.inverse_transform(rc_idx)


Phase 2 Inference (Sub-Category, Conditioned)

In [ ]:
subcat_probs = subcat_clf.predict_proba(X_emb)

subcat_preds = []
subcat_confs = []

for i in range(len(df)):
    if df.loc[i, "rc_confidence"] < RC_THRESHOLD:
        subcat_preds.append(None)
        subcat_confs.append(0.0)
        continue

    rc = df.loc[i, "predicted_resolution_code"]
    allowed_subcats = rescode_to_subcat.get(rc, [])

    allowed_idx = [
        subcat_le.transform([s])[0]
        for s in allowed_subcats
        if s in subcat_le.classes_
    ]

    probs = subcat_probs[i]
    mask = np.zeros_like(probs)
    mask[allowed_idx] = 1

    filtered = probs * mask
    if filtered.sum() == 0:
        subcat_preds.append(None)
        subcat_confs.append(0.0)
        continue

    filtered /= filtered.sum()
    idx = np.argmax(filtered)

    subcat_preds.append(subcat_le.inverse_transform([idx])[0])
    subcat_confs.append(filtered[idx])

df["predicted_sub_category"] = subcat_preds
df["subcat_confidence"] = subcat_confs


Phase 3 Inference (Fault Category, Conditioned)

In [ ]:
fault_probs = fault_clf.predict_proba(X_emb)

fault_preds = []
fault_confs = []

for i in range(len(df)):
    if (
        df.loc[i, "rc_confidence"] < RC_THRESHOLD or
        df.loc[i, "subcat_confidence"] < SUBCAT_THRESHOLD
    ):
        fault_preds.append(None)
        fault_confs.append(0.0)
        continue

    key = (
        df.loc[i, "predicted_resolution_code"] + "|" +
        df.loc[i, "predicted_sub_category"]
    )

    allowed_faults = rc_subcat_to_fault.get(key, [])

    allowed_idx = [
        fault_le.transform([f])[0]
        for f in allowed_faults
        if f in fault_le.classes_
    ]

    probs = fault_probs[i]
    mask = np.zeros_like(probs)
    mask[allowed_idx] = 1

    filtered = probs * mask
    if filtered.sum() == 0:
        fault_preds.append(None)
        fault_confs.append(0.0)
        continue

    filtered /= filtered.sum()
    idx = np.argmax(filtered)

    fault_preds.append(fault_le.inverse_transform([idx])[0])
    fault_confs.append(filtered[idx])

df["predicted_fault_category"] = fault_preds
df["fault_confidence"] = fault_confs


Final Decision & Review Reason

In [ ]:
decisions = []
reasons = []

for i in range(len(df)):
    if df.loc[i, "rc_confidence"] < RC_THRESHOLD:
        decisions.append("NEEDS_REVIEW")
        reasons.append("RC_LOW_CONF")

    elif df.loc[i, "subcat_confidence"] < SUBCAT_THRESHOLD:
        decisions.append("NEEDS_REVIEW")
        reasons.append("SUBCAT_LOW_CONF")

    elif df.loc[i, "fault_confidence"] < FAULT_THRESHOLD:
        decisions.append("NEEDS_REVIEW")
        reasons.append("FAULT_LOW_CONF")

    else:
        decisions.append("AUTO")
        reasons.append("CONFIDENT")

df["decision"] = decisions
df["review_reason"] = reasons


Export Output

In [ ]:
OUTPUT_PATH = r"D:\ML\SM Modeling\Data\Output\incident_predictions.xlsx"

df.to_excel(OUTPUT_PATH, index=False)

print("Inference completed. Output saved.")
